In [1]:
#!pip install xarray rioxarray geopandas matplotlib numpy netCDF4 scipy

## Configuration

In [2]:
# Everything you might want to change lives here.
# Swap REGION, tweak the break percentiles, adjust smoothing —
# then re-run from Cell 3 downward. No need to touch any other cell.

# Data files — download from https://www.climatologylab.org/terraclimate.html
# Pick a year, select variables tmax and ppt, download as NetCDF
TMAX_FILE = 'TerraClimate_tmax_2025.nc'
PPT_FILE  = 'TerraClimate_ppt_2025.nc'
BORDERS   = 'ne_10m_admin_0_countries.zip'  # Natural Earth 1:10m countries

# ── REGIONS TO PROCESS ───────────────────────────────────────────────────────
# Define regions as a list of dictionaries with 'name' and 'region' keys
# This allows processing multiple regions in one go
REGIONS_TO_PROCESS = [
    {'name': 'North America', 'region': 'North America'},
    {'name': 'USA', 'region': ["United States of America"]}
]

# ── COLOR & SMOOTHING ─────────────────────────────────────────────────────────
N     = 4     # palette grid size — 4×4 = 16 colors, matches the reference
SIGMA = 2.0   # Gaussian smoothing strength — higher = smoother blobs

# ── QUANTILE BREAK PERCENTILES ────────────────────────────────────────────────
# These control the orange/blue balance.
# Equal bins would be [0, 25, 50, 75, 100] — each bin gets 25% of pixels.
# We skew them so more pixels fall into warm/dry bins → more orange, less teal.
#
# Temp: top 30% of pixels are 'hot' (above the 70th percentile)
# Ppt:  top 10% of pixels are 'very wet' (above the 90th percentile)
TEMP_BREAKS = [0, 20, 45, 70, 100]
PPT_BREAKS  = [0, 60, 78, 90, 100]

# ── PALETTE ───────────────────────────────────────────────────────────────────
# 4×4 color matrix — hex codes sampled directly from the reference image.
# Layout:
#   row 0 = high precipitation (top of legend)    → teal / blue-green
#   row 3 = low  precipitation (bottom of legend) → grey / peach / orange
#   col 0 = low  temperature   (left of legend)   → cool tones
#   col 3 = high temperature   (right of legend)  → warm tones
PALETTE = [
    ['#3a9dbf', '#2a7d82', '#1e6050', '#164427'],  # high precip
    ['#7ab5c8', '#6d9490', '#5e7a62', '#4a5a30'],
    ['#b0cdd6', '#a8b5aa', '#9e9278', '#9a6e3a'],
    ['#d8d0c8', '#dbbfa4', '#dc9e72', '#dd6a2b'],  # low precip
]

FIGSIZE = (8, 8)

### Imports

In [3]:
# Four libraries do the heavy lifting:
#   xarray     — reads NetCDF files (the standard format for gridded climate data)
#   rioxarray  — adds spatial clipping and CRS support to xarray DataArrays
#   geopandas  — loads country/continent boundaries as vector polygons
#   scipy      — provides the Gaussian filter for smoothing
#
# Run this once to install if needed:
# !pip install xarray rioxarray geopandas matplotlib numpy netCDF4 scipy

import numpy as np
import xarray as xr
import rioxarray
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.ndimage import gaussian_filter

print('All imports OK')

All imports OK


### Load Data

In [4]:
# Load the base climate data once (shared for all regions)
tmax_da = xr.open_dataset(TMAX_FILE, engine='netcdf4')['tmax'].mean('time')
ppt_da  = xr.open_dataset(PPT_FILE,  engine='netcdf4')['ppt'].sum('time')

tmax_da = tmax_da.rio.write_crs('EPSG:4326')
ppt_da  = ppt_da.rio.write_crs('EPSG:4326')

# Load country boundaries once
world = gpd.read_file(BORDERS)

print('Data loading complete')

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\GTMSUA\\Employment\\Projects\\TerraClimatedata\\TerraClimate_tmax_2025.nc'

### Bivariate Palette & Helper Functions

In [ ]:
def hex_to_rgb(h):
    '''Convert a hex color string to a normalised (r, g, b) tuple.'''
    h = h.lstrip('#')
    return tuple(int(h[i:i+2], 16) / 255.0 for i in (0, 2, 4))

# Build the flat colormap — order must match bi_grid = ppt_q * N + temp_q
flat_colors = [hex_to_rgb(PALETTE[N - 1 - pq][tq])
               for pq in range(N)   # pq=0 → dry → PALETTE row 3
               for tq in range(N)]  # tq=0 → cold → PALETTE col 0

cmap = mcolors.ListedColormap(flat_colors)

# Build the legend grid separately — row 0 at bottom = dry, row N-1 at top = wet
legend_grid = np.array([[hex_to_rgb(PALETTE[N - 1 - row][col])
                          for col in range(N)]
                         for row in range(N)])

# Quick sanity check — verify the four corners are correct
print('Corner check:')
print(f'  cold & dry  (idx  0): {flat_colors[0]}  → should be light grey')
print(f'  hot  & dry  (idx  3): {flat_colors[3]}  → should be orange')
print(f'  cold & wet  (idx 12): {flat_colors[12]} → should be teal/blue')
print(f'  hot  & wet  (idx 15): {flat_colors[15]} → should be dark green')

# Preview the color matrix
fig, ax = plt.subplots(figsize=(3, 3))
ax.imshow(legend_grid, origin='lower', aspect='equal')
ax.set_xlabel('Temperature →', fontsize=8)
ax.set_ylabel('Precipitation →', fontsize=8)
ax.set_xticks([]); ax.set_yticks([])
ax.set_title('Bivariate color matrix', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
def draw_legend(ax, grid_rgb, n, text_color='k', fontsize=6):
    '''Draw the 2D color legend with axis labels and corner annotations.'''
    ax.imshow(grid_rgb, origin='lower', aspect='equal',
              extent=[0, n, 0, n], interpolation='nearest')
    ax.set_xlabel('Temperature →', fontsize=fontsize, color=text_color, labelpad=2)
    ax.set_ylabel('Precipitation →', fontsize=fontsize, color=text_color, labelpad=2)
    ax.set_xticks([]); ax.set_yticks([])
    
    # Remove border
    for sp in ax.spines.values():
        sp.set_visible(False)
    
    # Corner labels — black on light corner, white on dark corners
    kw = dict(fontsize=fontsize - 1, ha='center', va='center',
              alpha=0.9, linespacing=1.3, transform=ax.transData)
    ax.text(0.5,     0.5,     'cold\n& dry', color='k', **kw)
    ax.text(n - 0.5, 0.5,     'hot\n& dry',  color='w', **kw)
    ax.text(0.5,     n - 0.5, 'cold\n& wet', color='w', **kw)
    ax.text(n - 0.5, n - 0.5, 'hot\n& wet',  color='w', **kw)

### Processing Function

In [ ]:
def process_region(region_name, region_spec, tmax_da, ppt_da, world):
    '''
    Process and generate a bivariate climate map for a specified region.
    
    Parameters:
    -----------
    region_name : str
        Name for output file and display
    region_spec : str or list
        Either a continent name (str) or list of country names
    tmax_da, ppt_da : xarray.DataArray
        Temperature and precipitation data
    world : GeoDataFrame
        Country boundaries from Natural Earth
    '''
    
    print(f'\n{'='*60}')
    print(f'Processing: {region_name}')
    print(f'{'='*60}')
    
    # ── Select region ────────────────────────────────────────────────
    if isinstance(region_spec, list):
        region_gdf = world[world['ADMIN'].isin(region_spec)]
        print(f'Countries: {region_spec}')
    elif region_spec == 'World':
        region_gdf = world.copy()
        print('Region: World')
    else:
        region_gdf = world[world['CONTINENT'] == region_spec]
        print(f'Continent: {region_spec} — {len(region_gdf)} countries')
    
    # ── Clip to region ───────────────────────────────────────────────
    tmax_clip = tmax_da.rio.clip(region_gdf.geometry, region_gdf.crs, drop=True)
    ppt_clip  = ppt_da.rio.clip(region_gdf.geometry, region_gdf.crs, drop=True)
    
    print(f'Grid size: {tmax_clip.shape}')
    print(f'Temp:   {float(tmax_clip.min()):.1f}°C → {float(tmax_clip.max()):.1f}°C')
    print(f'Precip: {float(ppt_clip.min()):.0f} → {float(ppt_clip.max()):.0f} mm/yr')
    
    # ── Classification & Smoothing ───────────────────────────────────
    temp_vals  = tmax_clip.values.flatten()
    ppt_vals   = ppt_clip.values.flatten()
    valid      = ~np.isnan(temp_vals) & ~np.isnan(ppt_vals)
    
    t_breaks = np.nanpercentile(temp_vals[valid], TEMP_BREAKS)
    p_breaks = np.nanpercentile(ppt_vals[valid],  PPT_BREAKS)
    
    def qbin(arr, breaks):
        '''Assign values to quantile bins [0, N-1].'''
        return np.clip(np.digitize(arr, breaks[1:-1]), 0, len(breaks) - 2)
    
    nan_mask = np.isnan(tmax_clip.values) | np.isnan(ppt_clip.values)
    
    def smooth(arr):
        '''Fill NaN with mean, apply Gaussian blur, return smoothed array.'''
        filled = np.where(nan_mask, np.nanmean(arr), arr)
        return gaussian_filter(filled, sigma=SIGMA)
    
    ts = smooth(tmax_clip.values).flatten()
    ps = smooth(ppt_clip.values).flatten()
    vm = ~nan_mask.flatten()
    
    tq = np.full(ts.shape, -1, dtype=int)
    pq = np.full(ps.shape, -1, dtype=int)
    tq[vm] = qbin(ts[vm], t_breaks)
    pq[vm] = qbin(ps[vm], p_breaks)
    
    bi_grid = np.where(~nan_mask,
                       pq.reshape(nan_mask.shape) * N + tq.reshape(nan_mask.shape),
                       np.nan)
    
    print(f'Grid shape: {bi_grid.shape}')
    print(f'Value range: {np.nanmin(bi_grid):.0f} → {np.nanmax(bi_grid):.0f}  (expected 0–{N**2-1})')
    print(f'Temp breaks:  {[f"{v:.1f}" for v in t_breaks]}')
    print(f'Precip breaks: {[f"{v:.0f}" for v in p_breaks]}')
    
    # ── Plot the Map ─────────────────────────────────────────────────
    lons   = tmax_clip.coords['lon'].values
    lats   = tmax_clip.coords['lat'].values
    extent = [lons.min(), lons.max(), lats.min(), lats.max()]
    bounds = region_gdf.total_bounds   # [minx, miny, maxx, maxy]
    pad    = max(bounds[2] - bounds[0], bounds[3] - bounds[1]) * 0.03
    
    # ── Small version ────────────────────────────────────────────────
    fig = plt.figure(figsize=FIGSIZE, facecolor='white')
    ax  = fig.add_axes([0.02, 0.04, 0.94, 0.88], facecolor='white')
    
    ax.imshow(bi_grid, cmap=cmap, vmin=0, vmax=N**2 - 1,
              extent=extent, origin='upper', aspect='auto',
              interpolation='bilinear', zorder=1)
    
    region_gdf.boundary.plot(ax=ax, linewidth=0.2, color='white', zorder=2)
    
    ax.set_xlim(bounds[0] - pad, bounds[2] + pad)
    ax.set_ylim(bounds[1] - pad, bounds[3] + pad)
    ax.axis('off')
    
    title = region_name if isinstance(region_name, str) else ', '.join(region_name)
    ax.set_title(f'Temperature × Precipitation — {title}',
                 color='k', fontsize=13, fontweight='bold', pad=10, loc='left')
    
    ax_leg = fig.add_axes([0.08, 0.4, 0.15, 0.1], facecolor='white')
    draw_legend(ax_leg, legend_grid, N)
    
    fig.text(0.98, 0.02, 'Data: TerraClimate 2025 | @milanjanosov',
             ha='right', va='bottom', fontsize=7, color='#888888')
    
    fname = region_name.lower().replace(' ', '_').replace(',', '')
    plt.savefig(f'bivariate_{fname}_small.png', dpi=200,
                bbox_inches='tight', facecolor='white')
    plt.show()
    print(f'Saved: bivariate_{fname}_small.png')
    
    # ── Large version ────────────────────────────────────────────────
    fig = plt.figure(figsize=(16, 10), facecolor='white')
    ax = fig.add_axes([0.02, 0.05, 0.78, 0.88], facecolor='white')
    
    ax.imshow(bi_grid, cmap=cmap, vmin=0, vmax=N**2 - 1,
              extent=extent, origin='upper', aspect='auto',
              interpolation='bilinear', zorder=1)
    
    region_gdf.boundary.plot(ax=ax, linewidth=0.2, color='white', zorder=2)
    
    ax.set_xlim(bounds[0] - pad, bounds[2] + pad)
    ax.set_ylim(bounds[1] - pad, bounds[3] + pad)
    ax.axis('off')
    
    ax.set_title(f'Temperature × Precipitation — {title}',
                 color='k', fontsize=13, fontweight='bold', pad=10, loc='left')
    
    ax_leg = fig.add_axes([0.82, 0.35, 0.13, 0.30], facecolor='white')
    draw_legend(ax_leg, legend_grid, N, fontsize=8)
    ax_leg.set_aspect('equal', adjustable='box')
    
    fig.text(0.98, 0.02, 'Data: TerraClimate 2025 | @milanjanosov',
             ha='right', va='bottom', fontsize=7, color='#888888')
    
    plt.savefig(f'bivariate_{fname}_large.png', dpi=200,
                bbox_inches='tight', facecolor='white')
    plt.show()
    print(f'Saved: bivariate_{fname}_large.png')

### Generate All Maps

In [ ]:
# Process all regions
for region_config in REGIONS_TO_PROCESS:
    process_region(
        region_name=region_config['name'],
        region_spec=region_config['region'],
        tmax_da=tmax_da,
        ppt_da=ppt_da,
        world=world
    )

print('\n' + '='*60)
print('All maps generated successfully!')
print('='*60)